In [29]:
from ultralytics import YOLO
import glob
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import random
import numpy as np

In [1]:
DATA_YAML_FILEPATH = '/home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/assignment_4/dataset/vehicles/data.yaml'
DEVICE = 1

## Training

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data = DATA_YAML_FILEPATH, 
    epochs = 15,
    imgsz = 640,
    batch = 32, 
    workers=8,
    project='runs',
    name='vehicles_yolov8n',
    device = DEVICE
)

print(f"Weights saved at: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.33 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:1 (NVIDIA RTX A6000, 48517MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/assignment_4/dataset/vehicles/data.yaml, degrees=0.0, deterministic=True, device=1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vehicles_yolov8n, nbs=64, nms=Fal

## Evaluation on the held out test set

In [ ]:
# Load your trained model
TRAINED_MODEL_WEIGHTS = '/home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/runs/detect/assignment_4/runs/vehicles_yolov8n/weights/best.pt'
model = YOLO(TRAINED_MODEL_WEIGHTS)

# Validate on the test set
metrics = model.val(
    data = DATA_YAML_FILEPATH,
    split = 'test',
    conf = 0.25,
    iou = 0.6,
    imgsz = 640,
    device = DEVICE,
    workers = 8,
    plots = True,
    verbose = True
)

print()
print("*" * 45)
print("       TEST SET EVALUATION RESULTS")
print("*" * 45)
print(f"  mAP @ 0.5 IoU          : {metrics.box.map50:.4f}")
print(f"  mAP @ [0.5:0.95] IoU   : {metrics.box.map:.4f}")
print("*" * 45)

Ultralytics 8.4.33 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:1 (NVIDIA RTX A6000, 48517MiB)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 101.7±81.9 MB/s, size: 168.4 KB)
val: Scanning /home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/assignment_4/dataset/vehicles/test/labels... 3716 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3716/3716 2.0Kit/s 1.8s<0.0s
val: New cache created: /home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/assignment_4/dataset/vehicles/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 27, len(boxes) = 8102. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 233/233 6.5it/s 

### Qualitative Analysis

In [ ]:
CLASS_NAMES = ['bus', 'car', 'pickup', 'truck', 'van']
COLORS = ['red', 'blue', 'green', 'orange', 'purple']
DATASET_ROOT = '/home/bitupan/bitupan/aip_project/E9_246_Advanced_Image_Processing/assignment_4/dataset'
random.seed(42)

test_images = glob.glob(os.path.join(DATASET_ROOT, 'vehicles', 'test', 'images', '*'))
sample_imgs = random.sample(test_images, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    preds = model.predict(img_path, conf=0.25, iou=0.6, device=DEVICE, verbose=False)
    result = preds[0]

    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)

    boxes = result.boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        color    = COLORS[cls_id]

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 4,
            f"{CLASS_NAMES[cls_id]} {conf_val:.2f}",
            color='white', fontsize=8, backgroundcolor=color
        )

    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

plt.suptitle('YOLOv8n — Test Set Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/q3_qualitative_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1800x1000 with 6 Axes>

## Domain Shift and Failure Analysis

In [ ]:
CLASS_NAMES = ['bus', 'car', 'pickup', 'truck', 'van']
COLORS      = ['red', 'blue', 'green', 'orange', 'purple']
trained_model = YOLO(TRAINED_MODEL_WEIGHTS)

CAMPUS_VEHICLES_DIR = 'campus_vehicles'
campus_vehicles = sorted(glob.glob(os.path.join(DATASET_ROOT, CAMPUS_VEHICLES_DIR, '*.jpg')) + glob.glob(os.path.join(DATASET_ROOT, CAMPUS_VEHICLES_DIR, '*.jpeg')))

n = len(campus_vehicles)
print(n)

14


In [30]:
ncols = min(n, 3);  nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 6*nrows))
axes = np.array(axes).flatten()

print(f"{'Image':<35s} {'Detections':>11}  {'Classes found':<30s}  {'Conf (min/mean/max)'}")
print("-" * 100)

for idx, img_path in enumerate(campus_vehicles):
    preds  = model.predict(img_path, conf=0.1, iou=0.6, device=DEVICE, verbose=False)
    result = preds[0]
    # img    = Image.open(img_path)
    # img    = ImageOps.exif_transpose(img).convert('RGB')
    img = result.orig_img[:, :, ::-1] # convert BGR to RGB
    ax     = axes[idx]
    ax.imshow(img)

    boxes_xyxy = []
    confs      = []

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        color    = COLORS[cls_id]

        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{CLASS_NAMES[cls_id]} {conf_val:.2f}",
                color='white', fontsize=9, backgroundcolor=color,
                verticalalignment='bottom')

        boxes_xyxy.append([x1, y1, x2, y2])
        confs.append(conf_val)

    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

    # ── Pairwise IoU between predicted boxes ─────────────────────────────────
    n_boxes = len(boxes_xyxy)
    classes_found = list({CLASS_NAMES[int(b.cls[0])] for b in result.boxes})
    conf_str = (f"{min(confs):.2f} / {np.mean(confs):.2f} / {max(confs):.2f}"
                if confs else "—")


    print(f"{os.path.basename(img_path):<35s} {n_boxes:>11d}  "
          f"{str(classes_found):<30s}  conf {conf_str}")

# Hide unused axes
for ax in axes[n:]:
    ax.axis('off')

plt.suptitle("YOLOv8n — Predictions on Campus Vehicles (conf ≥ 0.1)", fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/q3_predictions_on_campus_vehicles.png', dpi=150, bbox_inches='tight')
plt.show()

Image                                Detections  Classes found                   Conf (min/mean/max)
----------------------------------------------------------------------------------------------------
IMG_20260402_173640057.jpg                    2  ['car']                         conf 0.14 / 0.36 / 0.57
IMG_20260402_182708955.jpg                    1  ['car']                         conf 0.78 / 0.78 / 0.78
IMG_20260402_183009057.jpg                    1  ['car']                         conf 0.91 / 0.91 / 0.91
IMG_20260402_183024592.jpg                    3  ['car']                         conf 0.15 / 0.36 / 0.71
IMG_20260402_183048412.jpg                    3  ['car']                         conf 0.27 / 0.36 / 0.54
IMG_5661.jpeg                                 3  ['car']                         conf 0.14 / 0.28 / 0.40
IMG_5662.jpeg                                 1  ['car']                         conf 0.83 / 0.83 / 0.83
IMG_5664.jpeg                                 1  ['car']       

<Figure size 2100x3000 with 15 Axes>